# Create node embeddings feature groups.

Up until now we use feature engineering, feature store and model training to create node embedding. We will now materialise this as node embeddings feature group. This feature group will be used to train anomaly detection model.

![Feature Stores](./images/online_offline_fs.png)

---
**NOTE**: 

In real life scenarios financial transaction are dynamically evolving graphs. If live Transaction Monitoring System is based on graph or node embeddings then this will require 1st to update the graph and node representations after new transactions arrive. Recomputing entire graph for every newly arrived transaction will lead to unaxeptable delayes and even monitoring system failures. This problem  will be more sever if large amount of updates happen in a short time window.

Contact us at Logical Clocks and we will help you to setup end to end graph based deep anomaly detection live Transaction Monitoring Systems. 

---

## Query Model Repository for best node embeddings model

In [ ]:
# Setup for local execution
import os
import json
import pandas as pd
import numpy as np

# Define paths
BASE_PATH = os.path.dirname(os.path.abspath("__file__"))
TRAINING_DATA_PATH = os.path.join(BASE_PATH, "training_data")
OUTPUT_PATH = os.path.join(BASE_PATH, "output")
MODELS_PATH = os.path.join(BASE_PATH, "models")
RESOURCES_PATH = os.path.join(BASE_PATH, "Resources")

# Override paths when running via pipeline (artifacts_dir injected by papermill)
try:
    if artifacts_dir:
        TRAINING_DATA_PATH = os.path.join(artifacts_dir, "data")
        OUTPUT_PATH = os.path.join(artifacts_dir, "data")
        MODELS_PATH = os.path.join(artifacts_dir, "models")
except NameError:
    pass

print(f"Training data: {TRAINING_DATA_PATH}")
print(f"Output: {OUTPUT_PATH}")

In [2]:
# Find the latest model directory
model_dirs = [d for d in os.listdir(MODELS_PATH) if d.startswith('node_embeddings_')]
if model_dirs:
    latest_model_dir = os.path.join(MODELS_PATH, sorted(model_dirs)[-1])
    print(f"Found model: {latest_model_dir}")
    
    # Load metadata
    with open(os.path.join(latest_model_dir, 'metadata.json'), 'r') as f:
        metadata = json.load(f)
    print(f"Model metrics: {metadata['metrics']}")
    print(f"Hyperparameters: {metadata['hyperparameters']}")
else:
    print("No model found! Run notebook 4 first.")

Found model: /home/adnoman/projects/aml_gan/AMLend2end/models/node_embeddings_0cf6becf
Model metrics: {'accuracy': 0.8891156462585034}
Hyperparameters: {'walk_number': 2, 'walk_length': 2, 'emb_size': 32}


In [3]:
# Load node embeddings from notebook 4
embeddings_path = os.path.join(TRAINING_DATA_PATH, "node_embeddings.csv")
node_embeddings_df = pd.read_csv(embeddings_path)

print(f"Loaded embeddings shape: {node_embeddings_df.shape}")
node_embeddings_df.head()

Loaded embeddings shape: (7347, 33)


,node_id,emb_0,emb_1,emb_2,emb_3,emb_4,emb_5,emb_6,emb_7,emb_8,...,emb_22,emb_23,emb_24,emb_25,emb_26,emb_27,emb_28,emb_29,emb_30,emb_31
0,3aa9646b,0.016530,-0.010947,-0.000496,-0.000766,0.028229,-0.007027,0.013945,-0.009410,0.007632,...,-0.012160,-0.022671,-0.005034,-0.024899,-0.022038,-0.021844,-0.009677,0.020635,0.008849,-0.024740
1,1e46e726,0.006591,0.013344,-0.015142,0.026666,0.026163,0.028550,-0.030208,-0.028693,0.022439,...,-0.004233,-0.011421,0.019160,0.018987,0.021968,-0.011300,0.030992,0.027277,-0.012025,-0.010329
2,49203bc3,0.005050,0.002158,-0.025958,0.010061,-0.030157,0.024989,-0.020235,0.019330,-0.028948,...,0.009855,-0.000529,0.009421,-0.022867,0.015096,0.002769,0.014327,-0.019856,-0.015230,-0.025997
3,a74d1101,0.020242,0.017416,0.028806,0.027612,-0.000559,0.013644,-0.029239,0.025620,0.031111,...,-0.016263,0.002794,0.010684,-0.031003,0.005417,0.019303,-0.019357,0.029952,-0.028007,-0.023648
4,616d4505,0.029466,0.012671,-0.006395,0.021655,0.031388,-0.016556,0.018658,-0.017666,0.013849,...,-0.011222,-0.022778,-0.022365,0.010624,-0.014411,0.018756,0.009777,-0.020094,0.006874,-0.022809


## Define model and load wights 

In [4]:
# Get embedding columns
emb_cols = [c for c in node_embeddings_df.columns if c.startswith('emb_')]
print(f"Embedding dimensions: {len(emb_cols)}")

# Preview embeddings
node_embeddings_df[['node_id'] + emb_cols[:5]].head()

Embedding dimensions: 32


,node_id,emb_0,emb_1,emb_2,emb_3,emb_4
0,3aa9646b,0.016530,-0.010947,-0.000496,-0.000766,0.028229
1,1e46e726,0.006591,0.013344,-0.015142,0.026666,0.026163
2,49203bc3,0.005050,0.002158,-0.025958,0.010061,-0.030157
3,a74d1101,0.020242,0.017416,0.028806,0.027612,-0.000559
4,616d4505,0.029466,0.012671,-0.006395,0.021655,0.031388


## connect hsfs library and get fs handle

In [5]:
# Load alert nodes to join with embeddings
alert_nodes_df = pd.read_csv(os.path.join(TRAINING_DATA_PATH, "alert_nodes_td.csv"))
print(f"Alert nodes: {len(alert_nodes_df)}")
print(f"SAR nodes: {alert_nodes_df['is_sar'].sum()}")

Alert nodes: 7347
SAR nodes: 816


### Get node and edge traininhg dataset objects 

In [6]:
# Create embedding array column (for compatibility with original format)
node_embeddings_df['embedding'] = node_embeddings_df[emb_cols].values.tolist()

# Rename node_id to id for consistency
node_embeddings_df = node_embeddings_df.rename(columns={'node_id': 'id'})

# Select final columns
node_embeddings_final = node_embeddings_df[['id', 'embedding']].copy()
print(f"Final embeddings shape: {node_embeddings_final.shape}")
node_embeddings_final.head()

Final embeddings shape: (7347, 2)


,id,embedding
0,3aa9646b,"[0.016529942, -0.010946558, -0.000495509, -0.0..."
1,1e46e726,"[0.00659081, 0.0133438045, -0.01514227, 0.0266..."
2,49203bc3,"[0.005049932, 0.0021584025, -0.025957875, 0.01..."
3,a74d1101,"[0.02024162, 0.017416382, 0.028806383, 0.02761..."
4,616d4505,"[0.029466301, 0.012670632, -0.0063951407, 0.02..."


### Read training datasets as pandas df 

In [7]:
# Join embeddings with alert nodes info
embeddings_with_labels = node_embeddings_df.merge(
    alert_nodes_df[['id', 'is_sar']], 
    on='id', 
    how='left'
)
embeddings_with_labels['is_sar'] = embeddings_with_labels['is_sar'].fillna(0).astype(int)

print(f"Embeddings with labels: {embeddings_with_labels.shape}")
print(f"SAR nodes in embeddings: {embeddings_with_labels['is_sar'].sum()}")

Embeddings with labels: (7347, 35)
SAR nodes in embeddings: 816


### Read hyperparamenter for graph embeddings

In [8]:
# Preview the data
print("Sample of embeddings with SAR labels:")
embeddings_with_labels[['id', 'is_sar'] + emb_cols[:3]].head(10)

Sample of embeddings with SAR labels:


,id,is_sar,emb_0,emb_1,emb_2
0,3aa9646b,0,0.016530,-0.010947,-0.000496
1,1e46e726,0,0.006591,0.013344,-0.015142
2,49203bc3,0,0.005050,0.002158,-0.025958
3,a74d1101,1,0.020242,0.017416,0.028806
4,616d4505,0,0.029466,0.012671,-0.006395
5,99af2455,1,0.013086,0.014228,0.026677
6,39be1ea2,0,0.022511,0.030015,0.014792
7,e7ec7bdb,1,-0.015121,-0.014387,-0.018982
8,e2e0d938,0,0.030832,-0.016196,0.022996
9,afc399a9,0,0.011579,-0.010085,-0.004988


### Construct stellargraph Graph object

In [9]:
# Statistics
print("Embedding statistics:")
print(f"  Total nodes: {len(embeddings_with_labels)}")
print(f"  SAR nodes (is_sar=1): {embeddings_with_labels['is_sar'].sum()}")
print(f"  Non-SAR nodes (is_sar=0): {(embeddings_with_labels['is_sar']==0).sum()}")
print(f"  Embedding dimensions: {len(emb_cols)}")

Embedding statistics:
  Total nodes: 7347
  SAR nodes (is_sar=1): 816
  Non-SAR nodes (is_sar=0): 6531
  Embedding dimensions: 32


### infer node embeddings

In [10]:
# Prepare final feature group data
# Keep id, all embedding columns, and is_sar
final_cols = ['id'] + emb_cols + ['is_sar']
node_embeddings_fg_df = embeddings_with_labels[final_cols].copy()

print(f"Feature group shape: {node_embeddings_fg_df.shape}")
node_embeddings_fg_df.head()

Feature group shape: (7347, 34)


,id,emb_0,emb_1,emb_2,emb_3,emb_4,emb_5,emb_6,emb_7,emb_8,...,emb_23,emb_24,emb_25,emb_26,emb_27,emb_28,emb_29,emb_30,emb_31,is_sar
0,3aa9646b,0.016530,-0.010947,-0.000496,-0.000766,0.028229,-0.007027,0.013945,-0.009410,0.007632,...,-0.022671,-0.005034,-0.024899,-0.022038,-0.021844,-0.009677,0.020635,0.008849,-0.024740,0
1,1e46e726,0.006591,0.013344,-0.015142,0.026666,0.026163,0.028550,-0.030208,-0.028693,0.022439,...,-0.011421,0.019160,0.018987,0.021968,-0.011300,0.030992,0.027277,-0.012025,-0.010329,0
2,49203bc3,0.005050,0.002158,-0.025958,0.010061,-0.030157,0.024989,-0.020235,0.019330,-0.028948,...,-0.000529,0.009421,-0.022867,0.015096,0.002769,0.014327,-0.019856,-0.015230,-0.025997,0
3,a74d1101,0.020242,0.017416,0.028806,0.027612,-0.000559,0.013644,-0.029239,0.025620,0.031111,...,0.002794,0.010684,-0.031003,0.005417,0.019303,-0.019357,0.029952,-0.028007,-0.023648,1
4,616d4505,0.029466,0.012671,-0.006395,0.021655,0.031388,-0.016556,0.018658,-0.017666,0.013849,...,-0.022778,-0.022365,0.010624,-0.014411,0.018756,0.009777,-0.020094,0.006874,-0.022809,0


In [ ]:
# Dummy cell - removed pyspark code

In [ ]:
# Dummy cell - removed pyspark code

In [ ]:
# Dummy cell - removed pyspark code

In [ ]:
# Dummy cell - removed pyspark code

In [ ]:
# Dummy cell - removed pyspark code

## Create embeddings feature group

In [11]:
# Save node embeddings feature group locally (replaces hsfs)
fg_path = os.path.join(OUTPUT_PATH, "node_embeddings_fg.parquet")
node_embeddings_fg_df.to_parquet(fg_path, index=False)
print(f"Saved node embeddings feature group to: {fg_path}")

# Also save as CSV for easier inspection
csv_path = os.path.join(OUTPUT_PATH, "node_embeddings_fg.csv")
node_embeddings_fg_df.to_csv(csv_path, index=False)
print(f"Saved CSV version to: {csv_path}")

Saved node embeddings feature group to: /home/adnoman/projects/aml_gan/AMLend2end/output/node_embeddings_fg.parquet
Saved CSV version to: /home/adnoman/projects/aml_gan/AMLend2end/output/node_embeddings_fg.csv


In [12]:
# Summary
print("=" * 50)
print("Node Embeddings Feature Group Created")
print("=" * 50)
print(f"Total nodes: {len(node_embeddings_fg_df)}")
print(f"Embedding dimensions: {len(emb_cols)}")
print(f"SAR nodes: {node_embeddings_fg_df['is_sar'].sum()}")
print(f"Non-SAR nodes: {(node_embeddings_fg_df['is_sar']==0).sum()}")
print(f"\nSaved to: {fg_path}")
print("=" * 50)

Node Embeddings Feature Group Created
Total nodes: 7347
Embedding dimensions: 32
SAR nodes: 816
Non-SAR nodes: 6531

Saved to: /home/adnoman/projects/aml_gan/AMLend2end/output/node_embeddings_fg.parquet


## Feature group provenance
![Feature group provenance](./images/provenance_fg.png)

In [13]:
# Done!